# Scatter3d plot with slider

In this example, we combine a three-dimensional scatter plot with a slider,
which is used to navigate the time dimension of our data.

In [ ]:
import plopp as pp
import scipp as sc
import numpy as np
import ipywidgets as ipw

We first generate some data that represents events detected on a cylindrical detector panel,
as a function of time.

In [ ]:
nphi = 100
nz = 20
nt = 50

r = sc.scalar(10.0, unit='m')
phi = sc.linspace('phi', 0, np.pi, nphi, unit='rad')
z = sc.linspace('z', -3.0, 3.0, nz, unit='m')
t = sc.linspace('time', 0., 6., 50, unit='rad')

x = r * sc.cos(phi)
y = r * sc.sin(phi)
xy = sc.sqrt(x**2 + (y - r)**2 + z**2)
xy.unit = 'rad'
a = sc.sin(xy)

da = sc.DataArray(data=a * sc.cos(t),
                  coords={'x': sc.broadcast(x, sizes=a.sizes),
                          'y': sc.broadcast(z, sizes=a.sizes),
                          'z': sc.broadcast(y, sizes=a.sizes),
                          't': t}
                 )
da

We then construct our interface with a slider,
a node that slices (and flattens) our data at the index of the slider,
and a `figure3d`.

In [ ]:
in_node = pp.input_node(da)
in_node.name = 'Input node'

# Make a slider
slider = ipw.IntSlider(max=nt-1, description='time',
                       layout={'width': '400px'})
slider_node = pp.widget_node(slider)
slider_node.name = 'Slider node'

slice_node = pp.node(
    lambda da, ind: da['time', ind].flatten(to='w'))(da=in_node, ind=slider_node)
slice_node.name = 'Slice by tof'

fig = pp.figure3d(slice_node, pixel_size=0.3)

pp.widgets.Box([fig, slider])

In [ ]:
pp.show_graph(in_node)